# Example MNL Model Estimations

## Disclaimer

This notebook is a simplified code template that complements the thesis by illustrating the estimation method. It is not the final code used in the research project. This final code can not be used as it contains data owned by Eneco.

The pilot choice sets used here did not include certainty in payback time as an attribute.

Certainty in payback time was only included in the final model and final choice data. The final data was purchased by Eneco and remains Eneco property, and therefore is not published in this repository.

This notebook only uses Pilot_Results_Update.csv and includes:
- basic linear MNL,
- fixed-parameter models,
- profitability model,
- interaction model,
- a simple non-linearity test (cost only).

Each code block is run once as an example.

In [1]:
from pathlib import Path
import pandas as pd

import biogeme.database as db
import biogeme.biogeme as bio
from biogeme import models
from biogeme.expressions import Beta, Variable, log

# Load only the approved publication dataset
raw = pd.read_csv(Path("Pilot_Results_Update.csv"), sep=";")

# Encode backup level as binary: Full house=1, One socket=0
backup_map = {"Full house": 1, "One socket": 0}

# Convert to Biogeme-ready long format with 3 alternatives (A, B, opt-out)
df = pd.DataFrame(
    {
        "ID": raw["ID"],
        "TASK_ID": raw["set"],
        "COST1": raw["A_cost"],
        "PAYBACK1": raw["A_payback"],
        "SELFCONSUMPTION1": raw["A_selfconsumption"],
        "BACKUP1": raw["A_backup"].map(backup_map),
        "COST2": raw["B_cost"],
        "PAYBACK2": raw["B_payback"],
        "SELFCONSUMPTION2": raw["B_selfconsumption"],
        "BACKUP2": raw["B_backup"].map(backup_map),
        "CHOICE": raw["Choice"] + 1,  # 0/1/2 -> 1/2/3
        "AV1": 1,
        "AV2": 1,
        "AV3": 1,
    }
)

biodata = db.Database("pilot_results_update", df)

# Biogeme variables
COST1 = Variable("COST1")
PAYBACK1 = Variable("PAYBACK1")
SELFCONSUMPTION1 = Variable("SELFCONSUMPTION1")
BACKUP1 = Variable("BACKUP1")

COST2 = Variable("COST2")
PAYBACK2 = Variable("PAYBACK2")
SELFCONSUMPTION2 = Variable("SELFCONSUMPTION2")
BACKUP2 = Variable("BACKUP2")

CHOICE = Variable("CHOICE")
AV = {1: Variable("AV1"), 2: Variable("AV2"), 3: Variable("AV3")}


def estimate_mnl(V, model_name):
    prob = models.logit(V, AV, CHOICE)
    ll = log(prob)
    biogeme = bio.BIOGEME(biodata, ll)
    biogeme.model_name = model_name
    biogeme.calculate_null_loglikelihood(AV)
    return biogeme.estimate()


print("Rows loaded:", len(df))

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


Rows loaded: 512


c:\python-projects\thesis\.venv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
# 1) Basic linear MNL
B_cost = Beta("B_cost", 0, None, None, 0)
B_payback = Beta("B_payback", 0, None, None, 0)
B_selfconsumption = Beta("B_selfconsumption", 0, None, None, 0)
B_backup = Beta("B_backup", 0, None, None, 0)
ASC_NO_BATT = Beta("ASC_NO_BATT", 0, None, None, 0)

V_linear = {
    1: B_cost * (COST1 / 1000) + B_payback * PAYBACK1 + B_selfconsumption * SELFCONSUMPTION1 + B_backup * BACKUP1,
    2: B_cost * (COST2 / 1000) + B_payback * PAYBACK2 + B_selfconsumption * SELFCONSUMPTION2 + B_backup * BACKUP2,
    3: ASC_NO_BATT,
}

results_linear = estimate_mnl(V_linear, "Linear-additive RUM-MNL battery")
print(results_linear.short_summary())
print(results_linear.get_beta_values())

Results for model Linear-additive RUM-MNL battery
Nbr of parameters:		5
Sample size:			512
Excluded data:			0
Null log likelihood:		-562.4895
Final log likelihood:		-483.9243
Likelihood ratio test (null):		157.1305
Rho square (null):			0.14
Rho bar square (null):			0.131
Akaike Information Criterion:	977.8485
Bayesian Information Criterion:	999.0402

{'B_cost': -0.1200442630323497, 'B_payback': -0.16825945703477324, 'B_selfconsumption': 0.03594524718136862, 'B_backup': 0.9341049062728797, 'ASC_NO_BATT': 0.2815181833545946}


In [3]:
# 2) Fixed models (one coefficient fixed to zero at a time)
fixed_params = ["B_cost", "B_payback", "B_selfconsumption", "B_backup"]
fixed_results = {}

for p in fixed_params:
    B_cost_f = Beta("B_cost", 0, None, None, int(p == "B_cost"))
    B_payback_f = Beta("B_payback", 0, None, None, int(p == "B_payback"))
    B_selfconsumption_f = Beta("B_selfconsumption", 0, None, None, int(p == "B_selfconsumption"))
    B_backup_f = Beta("B_backup", 0, None, None, int(p == "B_backup"))
    ASC_NO_BATT_f = Beta("ASC_NO_BATT", 0, None, None, 0)

    V_fixed = {
        1: B_cost_f * (COST1 / 1000) + B_payback_f * PAYBACK1 + B_selfconsumption_f * SELFCONSUMPTION1 + B_backup_f * BACKUP1,
        2: B_cost_f * (COST2 / 1000) + B_payback_f * PAYBACK2 + B_selfconsumption_f * SELFCONSUMPTION2 + B_backup_f * BACKUP2,
        3: ASC_NO_BATT_f,
    }

    r = estimate_mnl(V_fixed, f"Linear-additive RUM-MNL with {p} fixed to zero")
    fixed_results[p] = {
        "loglik": float(r.get_general_statistics()["Final log likelihood"]),
        "betas": r.get_beta_values(),
    }

print("Reference LL:", float(results_linear.get_general_statistics()["Final log likelihood"]))
for p in fixed_params:
    print(f"\nFixed: {p}")
    print("Final LL:", fixed_results[p]["loglik"])
    print(fixed_results[p]["betas"])

Reference LL: -483.9243

Fixed: B_cost
Final LL: -489.1576
{'B_payback': -0.13722331408629873, 'B_selfconsumption': 0.03326588944391767, 'B_backup': 0.7753872596082356, 'ASC_NO_BATT': 0.8700258736497221}

Fixed: B_payback
Final LL: -516.606
{'B_cost': 0.0033429911168869862, 'B_selfconsumption': 0.0285532900694578, 'B_backup': 0.45582178834013215, 'ASC_NO_BATT': 1.8279495780097061}

Fixed: B_selfconsumption
Final LL: -544.8489
{'B_cost': -0.009986535801881581, 'B_payback': -0.09288858177765393, 'B_backup': 0.4151720125247092, 'ASC_NO_BATT': -0.9396112263471251}

Fixed: B_backup
Final LL: -498.386
{'B_cost': -0.07184902866582642, 'B_payback': -0.1253157021035023, 'B_selfconsumption': 0.03199747107314301, 'ASC_NO_BATT': 0.20172640116116006}


In [4]:
# 3) Profitability model
# Profitability is defined as (cost in thousands) / payback.
B_profitability = Beta("B_profitability", 0, None, None, 0)
B_selfconsumption_p = Beta("B_selfconsumption", 0, None, None, 0)
B_backup_p = Beta("B_backup", 0, None, None, 0)
ASC_NO_BATT_p = Beta("ASC_NO_BATT", 0, None, None, 0)

V_profit = {
    1: B_profitability * ((COST1 / 1000) / PAYBACK1) + B_selfconsumption_p * SELFCONSUMPTION1 + B_backup_p * BACKUP1,
    2: B_profitability * ((COST2 / 1000) / PAYBACK2) + B_selfconsumption_p * SELFCONSUMPTION2 + B_backup_p * BACKUP2,
    3: ASC_NO_BATT_p,
}

results_profit = estimate_mnl(V_profit, "Profitability RUM-MNL battery")
print(results_profit.short_summary())
print(results_profit.get_beta_values())

Results for model Profitability RUM-MNL battery
Nbr of parameters:		4
Sample size:			512
Excluded data:			0
Null log likelihood:		-562.4895
Final log likelihood:		-507.2571
Likelihood ratio test (null):		110.4647
Rho square (null):			0.0982
Rho bar square (null):			0.0911
Akaike Information Criterion:	1022.514
Bayesian Information Criterion:	1039.468

{'B_profitability': 0.7374025427569684, 'B_selfconsumption': 0.02719330526195642, 'B_backup': 0.3976420043653756, 'ASC_NO_BATT': 2.0512666055105346}


In [5]:
# 4) Interaction model (cost, payback, and cost*payback)
B_cost_i = Beta("B_cost_int", 0, None, None, 0)
B_payback_i = Beta("B_payback_int", 0, None, None, 0)
B_cost_payback = Beta("B_cost_payback", 0, None, None, 0)
B_selfconsumption_i = Beta("B_selfconsumption_int", 0, None, None, 0)
B_backup_i = Beta("B_backup_int", 0, None, None, 0)
ASC_NO_BATT_i = Beta("ASC_NO_BATT_int", 0, None, None, 0)

V_interaction = {
    1: B_cost_i * (COST1 / 1000) + B_payback_i * PAYBACK1 + B_cost_payback * ((COST1 / 1000) * PAYBACK1) + B_selfconsumption_i * SELFCONSUMPTION1 + B_backup_i * BACKUP1,
    2: B_cost_i * (COST2 / 1000) + B_payback_i * PAYBACK2 + B_cost_payback * ((COST2 / 1000) * PAYBACK2) + B_selfconsumption_i * SELFCONSUMPTION2 + B_backup_i * BACKUP2,
    3: ASC_NO_BATT_i,
}

results_interaction = estimate_mnl(V_interaction, "Interaction RUM-MNL battery")
print(results_interaction.short_summary())
print(results_interaction.get_beta_values())

Results for model Interaction RUM-MNL battery
Nbr of parameters:		6
Sample size:			512
Excluded data:			0
Null log likelihood:		-562.4895
Final log likelihood:		-482.6628
Likelihood ratio test (null):		159.6533
Rho square (null):			0.142
Rho bar square (null):			0.131
Akaike Information Criterion:	977.3257
Bayesian Information Criterion:	1002.756

{'B_cost_int': -0.421438333853816, 'B_payback_int': -0.2978826991270047, 'B_cost_payback': 0.027879902014752534, 'B_selfconsumption_int': 0.04280627943370247, 'B_backup_int': 1.3245682842658084, 'ASC_NO_BATT_int': -0.5158036738086004}


In [6]:
# 5) Simple non-linearity test (cost only)
# Add a squared cost term and compare with the linear cost term.
from biogeme.results_processing import get_pandas_estimated_parameters

model_name_nl_cost = "Non-linear-additive RUM-MNL battery - costs"

B_cost_lin = Beta("B_cost_lin", 0, None, None, 0)
B_cost_sq = Beta("B_cost_sq", 0, None, None, 0)
B_payback_nl = Beta("B_payback", 0, None, None, 0)
B_selfconsumption_nl = Beta("B_selfconsumption", 0, None, None, 0)
B_backup_nl = Beta("B_backup", 0, None, None, 0)
ASC_NO_BATT_nl = Beta("ASC_NO_BATT", 0, None, None, 0)

V_nl_cost = {
    1: B_cost_lin * (COST1 / 1000) + B_cost_sq * ((COST1 / 1000) ** 2) + B_payback_nl * PAYBACK1 + B_selfconsumption_nl * SELFCONSUMPTION1 + B_backup_nl * BACKUP1,
    2: B_cost_lin * (COST2 / 1000) + B_cost_sq * ((COST2 / 1000) ** 2) + B_payback_nl * PAYBACK2 + B_selfconsumption_nl * SELFCONSUMPTION2 + B_backup_nl * BACKUP2,
    3: ASC_NO_BATT_nl,
}

results_nl_cost = estimate_mnl(V_nl_cost, model_name_nl_cost)
print(results_nl_cost.short_summary())
print(results_nl_cost.get_beta_values())
print(get_pandas_estimated_parameters(results_nl_cost))

Results for model Non-linear-additive RUM-MNL battery - costs
Nbr of parameters:		6
Sample size:			512
Excluded data:			0
Null log likelihood:		-562.4895
Final log likelihood:		-483.9241
Likelihood ratio test (null):		157.1307
Rho square (null):			0.14
Rho bar square (null):			0.129
Akaike Information Criterion:	979.8483
Bayesian Information Criterion:	1005.278

{'B_cost_lin': -0.11803505751139526, 'B_cost_sq': -0.00025953462320120143, 'B_payback': -0.1682623843116847, 'B_selfconsumption': 0.03594365014978534, 'B_backup': 0.9343692815290734, 'ASC_NO_BATT': 0.2838887904454397}
                Name     Value  Robust std err.  Robust t-stat.  \
0         B_cost_lin -0.118035         0.121170       -0.974124   
1          B_cost_sq -0.000260         0.015043       -0.017253   
2          B_payback -0.168262         0.023013       -7.311614   
3  B_selfconsumption  0.035944         0.003332       10.788235   
4           B_backup  0.934369         0.192314        4.858555   
5        ASC_NO